# Statistical Significance Testing
Wilcoxon signed-rank tests on per-fold MAE across cities and horizons.

Comparisons:
1. TabPFN vs TabPFN_NoWeather (RQ2)
2. TabPFN vs XGBoost (RQ1)
3. NeuralProphet vs NeuralProphet_NoWeather (RQ2 contrast)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
from itertools import product
from pathlib import Path

RESULTS_VERSION = "v6"
DATASETS = ["seoul", "london", "washington"]

COMPARISONS = [
    ("TabPFN",        "TabPFN_NoWeather"),   # RQ2
    ("TabPFN",        "XGBoost"),             # RQ1
    ("NeuralProphet", "NeuralProphet_NoWeather"),  # RQ2 contrast
]

df_detailed = pd.read_csv(f'../results/detailed_results_master_{RESULTS_VERSION}.csv')
df_detailed = df_detailed[df_detailed['weather_scenario'] == 'clean_only'].copy()
print(f"Loaded: {df_detailed.shape}")
print(f"Models: {sorted(df_detailed['model'].unique())}")
print(f"Folds:  {sorted(df_detailed['fold'].unique())}")

## 1. Per city × horizon tests
One Wilcoxon test per (city, horizon, comparison). Observations = per-fold MAE values.

In [ ]:
def filter_results(df, version, dataset):
    df = df[df['version'] == version].copy()
    df = df[df['dataset'] == dataset].copy()
    return df

rows = []

for dataset, (model_a, model_b) in product(DATASETS, COMPARISONS):
    df_city = filter_results(df_detailed, RESULTS_VERSION, dataset)
    horizons = sorted(df_city['horizon'].unique())

    for h in horizons:
        df_h = df_city[df_city['horizon'] == h]

        a_vals = df_h[df_h['model'] == model_a].sort_values('fold')['MAE'].values
        b_vals = df_h[df_h['model'] == model_b].sort_values('fold')['MAE'].values

        if len(a_vals) == 0 or len(b_vals) == 0:
            continue
        if len(a_vals) != len(b_vals):
            print(f"WARNING: unequal folds for {model_a} vs {model_b} in {dataset} h={h}")
            continue

        diff = a_vals - b_vals
        if np.all(diff == 0):
            stat, p = np.nan, np.nan
        else:
            stat, p = wilcoxon(a_vals, b_vals, alternative='two-sided')

        rows.append({
            'comparison': f"{model_a} vs {model_b}",
            'model_a': model_a,
            'model_b': model_b,
            'dataset': dataset,
            'horizon': h,
            'n_folds': len(a_vals),
            'mean_MAE_a': a_vals.mean().round(2),
            'mean_MAE_b': b_vals.mean().round(2),
            'mean_diff': (a_vals - b_vals).mean().round(2),
            'statistic': stat,
            'p_value': p,
        })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

## 2. Multiple comparison correction (Holm-Bonferroni)
Applied separately per comparison pair.

In [ ]:
from statsmodels.stats.multitest import multipletests

corrected_rows = []

for comp, grp in results_df.groupby('comparison'):
    grp = grp.copy().dropna(subset=['p_value'])
    reject, p_adj, _, _ = multipletests(grp['p_value'], method='holm')
    grp['p_adjusted'] = p_adj.round(4)
    grp['significant'] = reject
    corrected_rows.append(grp)

results_corrected = pd.concat(corrected_rows).reset_index(drop=True)

print(results_corrected[
    ['comparison','dataset','horizon','n_folds',
     'mean_MAE_a','mean_MAE_b','mean_diff',
     'p_value','p_adjusted','significant']
].to_string(index=False))

## 3. Pooled test (across all cities and horizons)
Single test per comparison pair using all fold-level observations pooled.

In [ ]:
for model_a, model_b in COMPARISONS:
    a_all = df_detailed[df_detailed['model'] == model_a].sort_values(
        ['dataset','horizon','fold'])['MAE'].values
    b_all = df_detailed[df_detailed['model'] == model_b].sort_values(
        ['dataset','horizon','fold'])['MAE'].values

    if len(a_all) == 0 or len(b_all) == 0:
        print(f"{model_a} vs {model_b}: one model missing, skipping")
        continue
    if len(a_all) != len(b_all):
        print(f"{model_a} vs {model_b}: unequal lengths ({len(a_all)} vs {len(b_all)}), skipping")
        continue

    stat, p = wilcoxon(a_all, b_all, alternative='two-sided')
    median_diff = np.median(a_all - b_all)
    direction = "A better" if median_diff < 0 else "B better"

    print(f"\n{model_a} vs {model_b}")
    print(f"  n pairs    : {len(a_all)}")
    print(f"  median diff: {median_diff:.2f} ({direction})")
    print(f"  statistic  : {stat:.1f}")
    print(f"  p-value    : {p:.4f}")
    print(f"  significant: {p < 0.05}")

## 4. Save results

In [ ]:
Path('../results/tables/cross_city').mkdir(parents=True, exist_ok=True)
results_corrected.to_csv(
    '../results/tables/cross_city/wilcoxon_results.csv', index=False
)
print("Saved: wilcoxon_results.csv")